In [ ]:
using System;
using System.Xml;

var path = "/usr/share/glib-2.0/schemas/org.fedorahosted.background-logo-extension.gschema.xml";
XmlDocument doc = new();
doc.Load(path);
var root = doc.DocumentElement;


In [ ]:
using System.Collections;
using System.Collections.Generic;
using System.Diagnostics;

public class GVariantParser
{
    private static Type FromChar(char c) => c switch
    {
        'b' => typeof(Boolean),
        'y' => typeof(Char),
        'n' => typeof(Int16),
        'q' => typeof(UInt16),
        'i' => typeof(Int32),
        'u' => typeof(UInt32),
        'x' => typeof(Int64),
        't' => typeof(UInt64),
        'h' => typeof(Int32),  // handle..?
        'd' => typeof(Double),
        's' or 'o' or 'g' => typeof(String),
        _ => throw new ArgumentOutOfRangeException(nameof(c), $"Not implemented GVariant representation: {c}"),
    };

    public static Type Parse(string repr)
    {
        Stack<Type> stack = new();
        Stack<Stack<Type>> metaStack = new();
        foreach (char c in repr.ToCharArray())
        {
            switch (c)
            {
                case '(':
                case '{':
                    metaStack.Push(stack);
                    stack = new();
                    break;
                case ')':
                case '}':
                    Type[] typeParams = stack.ToArray();
                    typeParams.Reverse();
                    stack = metaStack.Pop();
                    Type constructed = c switch
                    {
                        ')' => typeof(Tuple)
                            .GetMethods()
                            .Where(m => m.Name == "Create" && m.GetParameters().Count() == typeParams.Count())
                            .First()
                            .ReturnType
                            .GetGenericTypeDefinition()
                            .MakeGenericType(typeParams),
                        _ => typeof(Dictionary<,>)
                            .MakeGenericType(typeParams),
                    };
                    stack.Push(constructed);
                    break;

                default:
                    stack.Push(FromChar(c));
                    break;
            }
        }

        Type result = stack.Pop();
        Debug.Assert(stack.Count == 0, "We should have no types left on the stack");
        return result;
    }
}

GVariantParser.Parse("{si}")

System.Collections.Generic.Dictionary<System.Int32,System.String>

In [ ]:
foreach (XmlElement xmlSchema in root.SelectNodes("schema"))
{
    string id = xmlSchema.GetAttribute("id");
    string path = xmlSchema.GetAttribute("path");

    foreach (XmlElement xmlKey in xmlSchema.SelectNodes("key"))
    {
        string name = xmlKey.GetAttribute("name");
        string typeName = xmlKey.GetAttribute("type");
        Console.WriteLine(name);
        if (typeName is "")
        {
            string enumName = xmlKey.GetAttribute("enum");
            var values = root.SelectNodes($"enum[@id='{enumName}']/value/@nick");
        }
    }
}